# Chapter 9 — Escalation and Human-in-the-Loop · Homework (self-test)

**Goal:** check that you can decide *when* an agent hands off, *where* each rule belongs (prompt, code or human), and *how* to calibrate and audit human review.

- **Theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 1.5–2 hours
- **Estimated API cost:** $0. Everything scored runs offline. The one optional live exercise costs about $0.02 with the default model.
- **Prerequisites:** you have worked through the practice notebook.

### How to use this notebook

1. Attempt everything **without peeking**. Solutions and explanations are at the very bottom, inside collapsed blocks.
2. **Part A — Quiz:** 10 exam-style scenario questions. Put your letters in the `answers` cell and run the grader.
3. **Part B — Coding exercises:** 6 offline exercises. Replace each `TODO`, then run the check cell under it. Checks never crash the notebook; they print ✓ or a hint.
4. **Part C — Architecture scenario:** write a short design, then compare it with the rubric.
5. **Scorecard:** the overall score is the average of your quiz % and your exercise %. The pass mark is **72%**, which mirrors the Claude exam's 720/1000.

In [ ]:
import hashlib
import json
import math
import os
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Callable, Literal

import anthropic
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ConfigDict, ValidationError

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # only the optional live exercise uses it

REFUND_LIMIT_USD = 500.00


# ---- shared types (the README's, without gate_log) ----
@dataclass
class CaseState:
    verified_customer_id: str | None = None
    orders: dict[str, dict] = field(default_factory=dict)       # order_id -> record returned by lookup_order
    refunded: dict[str, float] = field(default_factory=dict)    # order_id -> amount already refunded
    threshold_denials: int = 0
    escalated: bool = False


class ActionTaken(BaseModel):
    model_config = ConfigDict(extra="forbid")
    step: str
    outcome: str


class HandoffSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")
    escalation_reason: Literal["customer_requested_human", "policy_gap",
                               "cannot_make_progress", "threshold_exceeded", "other"]
    reason_detail: str
    customer_id: str | None
    identity_verified: bool
    order_ids: list[str]
    issue_summary: str
    root_cause: str | None
    facts_verified: list[str]
    customer_claims: list[str]
    actions_taken: list[ActionTaken]
    offers_made: list[str]
    amount_at_stake_usd: float | None
    recommended_action: str
    recommendation_rationale: str
    urgency: Literal["low", "normal", "high"]
    open_questions: list[str]


# ---- grading helpers ----
RESULTS: dict[int, bool] = {}
QUIZ_RESULT: tuple[int, int] | None = None


def _run_check(n: int, checks: Callable[[], None]) -> None:
    """Run one exercise's asserts and record the result. Never raises."""
    try:
        checks()
    except NotImplementedError:
        RESULTS[n] = False
        print(f"… Exercise {n}: not implemented yet. Replace the TODO and rerun this cell.")
    except AssertionError as err:
        RESULTS[n] = False
        print(f"✗ Exercise {n} not passed yet: {err}")
    except Exception as err:  # a bug in the solution must not crash the notebook
        RESULTS[n] = False
        print(f"✗ Exercise {n} raised {type(err).__name__}: {err}")
    else:
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")


print("Setup done. Model for the optional live exercise:", MODEL)

In [ ]:
# Only the optional live exercise needs a key. Without one, everything else still runs.
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
print("key found: the optional live exercise can run" if HAS_KEY else "no key: the optional live exercise will be skipped")

## Part A — Quiz

Each question has exactly one best answer. Distractors are the anti-patterns from [Common mistakes](README.md#common-mistakes-and-anti-patterns).

**Q1.** A customer's first message is: "Before anything else, I want to talk to a human about my order ORD-4471." The agent has tools to look up orders and issue refunds. Which response best follows the escalation guidance?

- **A)** Look up ORD-4471 first, so the agent can try to solve the problem before involving a person.
- **B)** Ask the customer to describe the problem, so the agent can judge whether a human is really needed.
- **C)** Call `escalate_to_human` right away, put the order number the customer already gave into the handoff, and tell the customer a specialist will pick it up.
- **D)** Offer a discount code to calm things down, and escalate only if the customer asks a second time.

**Q2.** A verified customer asks for a replacement to be shipped to a hotel abroad because they are traveling. The written policy covers replacements sent to the address on the order and says nothing about other addresses. The agent's tools can technically change the shipping address. What should the agent do?

- **A)** Escalate as a policy gap, with the verified order details, the requested address and the open question in the handoff.
- **B)** Ship to the hotel, because the replacement itself is covered by policy.
- **C)** Refuse, because the policy only mentions the address on the order.
- **D)** Ask the model how likely a manager would be to approve it, and ship if it says above 80%.

**Q3.** Your harness blocks `process_refund` until `state.verified` is true. The flag is set when the assistant's text contains "identity verified". Red-teaming finds a prompt that makes the model write that phrase without calling `get_customer`, and the refund goes through. What is the right fix?

- **A)** Add few-shot examples showing that the model never writes "identity verified" before calling `get_customer`.
- **B)** Tighten the regular expression so only the exact sentence "Customer identity verified." sets the flag.
- **C)** Add a second model call that reviews every refund for evidence of verification before it executes.
- **D)** Set the gate's state only from `get_customer` tool results (status `found`, exactly one match), never from what the model writes.

**Q4.** Refunds above $500 are blocked in code, and the denial tells the model to call `escalate_to_human` with `threshold_exceeded`. Logs show that in about 3% of blocked conversations the model instead says "I'm unable to help with that" and ends its turn. Those customers never reach a person. What is the most reliable fix?

- **A)** Rewrite the denial message in capital letters so the model can't miss it.
- **B)** After the loop ends, check in code whether a threshold block happened without an escalation, and file the escalation automatically from the case state.
- **C)** Add more few-shot examples of escalating after a denial.
- **D)** Raise the refund limit so fewer conversations are blocked.

**Q5.** On Claude Managed Agents you define `issue_refund` as a custom tool and set `default_config.permission_policy: {"type": "always_ask"}` on the agent toolset. Refunds still execute without any confirmation step. Why, and what should you do?

- **A)** `always_ask` only takes effect in interactive sessions; switch the toolset to `auto`.
- **B)** Policies must be set per tool; add a `configs` entry `{"name": "issue_refund", "permission_policy": {"type": "always_ask"}}` to the agent toolset.
- **C)** Permission policies govern only server-executed tools (the agent toolset and MCP toolsets). Custom tools run in your application, so put the approval step in your handler for `agent.custom_tool_use` before you send the result.
- **D)** Custom tools can't be gated at all; move the refund into a bash command so the agent toolset's policy applies to it.

**Q6.** On 5,000 labeled invoices, the 0.90–0.95 confidence band was 99.3% accurate for `vendor_name` but 84% accurate for `tax_amount`. The team proposes one rule: auto-accept every field with confidence ≥ 0.90. What should you recommend?

- **A)** Accept the rule: a confidence of at least 0.9 means at least 90% accuracy for every field.
- **B)** Set a separate threshold per field (and per document type) from reliability tables with a minimum sample size, and keep `tax_amount` on review until some threshold meets the target.
- **C)** Raise the single threshold to 0.95 for every field.
- **D)** Instruct the model to report lower confidence for tax amounts.

**Q7.** Your team audited 50 randomly sampled auto-accepted handwritten receipts and found no errors. A manager wants to report that this segment's error rate is below 1%. What is the accurate statement?

- **A)** The error rate is 0%, because no errors were found.
- **B)** The error rate is below 1%, because 50 random samples is a statistically significant audit.
- **C)** The error rate is below 2%, because 1/50 = 2%.
- **D)** With 0 errors in 50, the 95% upper bound is about 3/50 = 6%. Supporting "below 1%" needs roughly 300 audited items in that stratum.

**Q8.** Your pipeline flags 2,000 fields a day for review, but the reviewers can handle 600. The daily stratified audit of auto-accepted items takes 60 of those slots. What is the best plan?

- **A)** Keep the 60 audit items and fill the other 540 slots in order of estimated probability of error × cost of that field being wrong; the rest waits.
- **B)** Pause the audit sample until the backlog clears, and review flagged fields first in, first out.
- **C)** Lower the calibrated thresholds so fewer fields are flagged.
- **D)** Review a random 600 of the 2,000 flagged fields, to avoid bias.

**Q9.** An invoice states a total of $4,980, but its line items sum to $5,180. The extraction model returns `total: 4980` with confidence 0.97, which is above the calibrated threshold for `total`. What should the pipeline do?

- **A)** Auto-accept it, because 0.97 is above the calibrated threshold.
- **B)** Recompute the sum in code, detect the conflict, and route the document to a reviewer with both values and the source snippets.
- **C)** Ask the model which value is more likely to be right and accept its choice.
- **D)** Re-run the extraction until the total and the line items agree.

**Q10.** A verified customer writes: "Please refund the $35 mug from ORD-1 that arrived chipped, and also add the 2,000 loyalty points your spring promotion promised me." Policy covers the mug refund. Only the loyalty team can adjust points, and the agent has no tool for it. What should the agent do?

- **A)** Escalate the whole conversation, so one person handles both requests.
- **B)** Refund the mug and tell the customer the points can't be added.
- **C)** Ask the customer to open a separate conversation about the points.
- **D)** Process the mug refund, then escalate only the points request, with a handoff that says the refund is already done.

In [ ]:
# Fill in one letter per question: "A", "B", "C" or "D".
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc",
    "q9": "b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e",
    "q10": "c72adbd855b96e33ffae9308bfc1d5cdfa7fb315301530c3b1f890435529ec04",
}  # sha256 of "qid:LETTER", so the answers are not readable here


def grade_quiz(answers: dict) -> None:
    global QUIZ_RESULT
    correct = 0
    for qid, expected_hash in ANSWER_KEY.items():
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) and given.strip() else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == expected_hash
        correct += ok
        print(f"{qid:>3}: {'✓' if ok else '✗'}" + ("  (unanswered)" if letter is None else ""))
    QUIZ_RESULT = (correct, len(ANSWER_KEY))
    print(f"\nQuiz score: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")


grade_quiz(answers)

## Part B — Coding exercises

All six run offline and use only plain Python, Pydantic and the shared types from the setup cell.

### Exercise 1 — An escalation trigger classifier

Implement `route_turn(turn) -> (decision, reason)` for the latest customer turn. `turn` is a dict of facts the agent knows; any key may be missing, so use these defaults:

| key | meaning | default |
|---|---|---|
| `asks_human` | the customer explicitly asked for a person | `False` |
| `matches` | accounts matching the customer (1 = exactly one, verified) | `1` |
| `id_requests` | how many times the agent already asked for an identifier | `0` |
| `policy_covers` | `True`, or `False`/`None` when the policy is silent or ambiguous | `True` |
| `gate_blocked` | code blocked an action as above the agent's authority | `False` |
| `tool_failures` | consecutive failures of a tool after its own retries | `0` |
| `rejected_offers` | valid resolutions the customer rejected | `0` |
| `sentiment`, `self_confidence` | mood (−1..1) and the model's 1–10 self-rating | — |

Apply the first rule that matches:

1. explicit request → `("escalate", "customer_requested_human")`
2. `matches != 1` and `id_requests >= 2` → `("escalate", "cannot_make_progress")`
3. `matches != 1` → `("ask", None)`
4. `policy_covers is not True` → `("escalate", "policy_gap")`
5. `gate_blocked` → `("escalate", "threshold_exceeded")`
6. `tool_failures >= 3` or `rejected_offers >= 2` → `("escalate", "cannot_make_progress")`
7. otherwise → `("resolve", None)`

`sentiment` and `self_confidence` must have **no** effect on the result.

[Theory: 9.1](README.md#91-when-to-escalate) · [9.3](README.md#93-resolving-ambiguity-ask-dont-guess)

In [ ]:
def route_turn(turn: dict) -> tuple[str, str | None]:
    # TODO: apply the seven rules in order, with the defaults from the table
    raise NotImplementedError

In [ ]:
def check_ex1():
    def checks():
        cases = [
            ("explicit request, nothing known yet", {"asks_human": True, "matches": 0}, ("escalate", "customer_requested_human")),
            ("explicit request beats a policy gap", {"asks_human": True, "policy_covers": None}, ("escalate", "customer_requested_human")),
            ("angry but covered (mood is not complexity)", {"sentiment": -0.95, "self_confidence": 2}, ("resolve", None)),
            ("calm, confident, policy silent", {"sentiment": 0.6, "self_confidence": 9, "policy_covers": None}, ("escalate", "policy_gap")),
            ("policy explicitly False", {"policy_covers": False}, ("escalate", "policy_gap")),
            ("three accounts match", {"matches": 3}, ("ask", None)),
            ("three accounts match, policy also silent: identity first", {"matches": 3, "policy_covers": None}, ("ask", None)),
            ("no account after two identifier requests", {"matches": 0, "id_requests": 2}, ("escalate", "cannot_make_progress")),
            ("one identifier request so far", {"matches": 0, "id_requests": 1}, ("ask", None)),
            ("gate blocked a large refund", {"gate_blocked": True}, ("escalate", "threshold_exceeded")),
            ("tool failed 3 times", {"tool_failures": 3}, ("escalate", "cannot_make_progress")),
            ("tool failed twice (not yet)", {"tool_failures": 2}, ("resolve", None)),
            ("two offers rejected", {"rejected_offers": 2}, ("escalate", "cannot_make_progress")),
            ("empty dict uses the defaults", {}, ("resolve", None)),
        ]
        for label, turn, expected in cases:
            got = route_turn(dict(turn))
            assert got is not None, "return a (decision, reason) tuple"
            assert tuple(got) == expected, f"{label}: expected {expected}, got {got}"
        for _, turn, expected in cases:
            for s, c in [(-1.0, 1), (1.0, 10), (-0.7, 3)]:
                got = tuple(route_turn({**turn, "sentiment": s, "self_confidence": c}))
                assert got == expected, (f"sentiment={s} / self_confidence={c} changed the decision for {turn} to {got}; "
                                         "mood and self-rated confidence must not be triggers")
    _run_check(1, checks)


check_ex1()

### Exercise 2 — A `get_customer` result that makes asking natural

Implement `customer_lookup_result(matches)`. `matches` is the list of CRM records your search returned; each record has `customer_id`, `name`, `email`, `phone` and `order_ids`. Return exactly:

- **one match:** `{"status": "found", "customer_id": ..., "name": ..., "order_ids": [...]}`
- **several:** `{"status": "multiple_matches", "count": n, "ask_for_one_of": ["email", "phone", "order_number"]}`. Nothing else: no candidate's ID, email or phone may appear anywhere in the result (they are other people's data, and the model would be tempted to pick one).
- **none:** `{"status": "not_found", "hint": "<a non-empty instruction for the agent>"}`

The result must be JSON-serializable and you must not mutate `matches`.

[Theory: 9.3](README.md#93-resolving-ambiguity-ask-dont-guess)

In [ ]:
def customer_lookup_result(matches: list[dict]) -> dict:
    # TODO: found / multiple_matches / not_found, with no candidate data leaking on multiple matches
    raise NotImplementedError

In [ ]:
def check_ex2():
    def checks():
        marias = [
            {"customer_id": "CUST-1", "name": "Maria Lopez", "email": "m1@example.com", "phone": "+1-555-0001", "order_ids": ["ORD-1"]},
            {"customer_id": "CUST-2", "name": "Maria Lopez", "email": "m2@example.org", "phone": "+1-555-0002", "order_ids": ["ORD-2", "ORD-3"]},
            {"customer_id": "CUST-3", "name": "Maria Lopez", "email": "m3@example.net", "phone": "+1-555-0003", "order_ids": []},
        ]
        snapshot = json.dumps(marias)

        one = customer_lookup_result(marias[1:2])
        assert isinstance(one, dict), "return a dict"
        assert one == {"status": "found", "customer_id": "CUST-2", "name": "Maria Lopez", "order_ids": ["ORD-2", "ORD-3"]}, \
            f"one match should return exactly status, customer_id, name, order_ids; got {one}"

        multi = customer_lookup_result(marias)
        assert multi.get("status") == "multiple_matches" and multi.get("count") == 3, f"three matches: got {multi}"
        assert multi.get("ask_for_one_of") == ["email", "phone", "order_number"], "list the identifiers that would disambiguate"
        blob = json.dumps(multi)
        leaked = [v for c in marias for v in (c["customer_id"], c["email"], c["phone"]) if v in blob]
        assert not leaked, f"multiple_matches leaks other customers' data: {leaked}. Return only the count and what to ask for."
        assert set(multi) == {"status", "count", "ask_for_one_of"}, f"unexpected keys in multiple_matches: {sorted(set(multi) - {'status', 'count', 'ask_for_one_of'})}"
        assert customer_lookup_result(marias[:2])["count"] == 2, "count must be the number of matches"

        none = customer_lookup_result([])
        assert none.get("status") == "not_found" and isinstance(none.get("hint"), str) and none["hint"].strip(), \
            "no match: status not_found plus a non-empty hint"
        assert json.dumps(marias) == snapshot, "do not mutate the matches list"
    _run_check(2, checks)


check_ex2()

### Exercise 3 — A prerequisite and threshold gate that fails closed

Implement three functions over the `CaseState` from the setup cell (the README's, without `gate_log`).

**`gate(tool, args, state) -> str | None`**, applied in this order. Return `None` to allow; otherwise a message that starts with `"BLOCKED"`.

1. `lookup_order` or `process_refund` while `state.verified_customer_id is None` → blocked; the message must mention `get_customer`.
2. `process_refund` on an order that is not in `state.orders`, or whose `customer_id` is not the verified customer → blocked.
3. `process_refund` with `amount <= 0`, or where `amount` plus what was already refunded on that order (`state.refunded`) exceeds the order's `total` → blocked.
4. `process_refund` where `amount` plus what was already refunded on that order exceeds `REFUND_LIMIT_USD` (a running total; exactly $500 in total is allowed) → increment `state.threshold_denials` and return a message that contains both `escalate_to_human` and `threshold_exceeded`, so the model knows what to do next.

**`record_result(tool, output, state)`** updates state from tool **results** only: `get_customer` with `status == "found"` sets the verified ID; `lookup_order` output with an `order_id` is stored in `state.orders`; `process_refund` output with `status == "refunded"` adds its `amount` to `state.refunded[order_id]`; any `escalate_to_human` result sets `state.escalated = True`.

**`close_case(state) -> dict | None`** runs after the loop. If there was a threshold denial and no escalation, set `state.escalated = True` and return `{"escalation_reason": "threshold_exceeded", "customer_id": <verified id>, "order_ids": <sorted order ids in state.orders>}`. Otherwise return `None`.

[Theory: 9.4](README.md#94-programmatic-enforcement-vs-prompt-guidance)

In [ ]:
def gate(tool: str, args: dict, state: CaseState) -> str | None:
    # TODO: rules 1-4 in order
    raise NotImplementedError


def record_result(tool: str, output: dict, state: CaseState) -> None:
    # TODO: update state from tool results only
    raise NotImplementedError


def close_case(state: CaseState) -> dict | None:
    # TODO: fail closed when a threshold block was not escalated
    raise NotImplementedError

In [ ]:
def check_ex3():
    def checks():
        order_a = {"order_id": "ORD-A", "customer_id": "C1", "total": 900.0}
        order_b = {"order_id": "ORD-B", "customer_id": "C2", "total": 50.0}

        s = CaseState()
        msg = gate("lookup_order", {"order_id": "ORD-A"}, s)
        assert isinstance(msg, str) and msg.startswith("BLOCKED") and "get_customer" in msg, \
            "rule 1: block lookup_order before verification with a message that mentions get_customer"
        assert gate("process_refund", {"order_id": "ORD-A", "amount": 10.0}, s), "rule 1: block process_refund before verification"
        assert gate("get_customer", {"email": "x@example.com"}, s) is None, "get_customer itself must always be allowed"

        record_result("get_customer", {"status": "multiple_matches", "count": 3}, s)
        assert s.verified_customer_id is None, "multiple_matches must NOT verify anyone"
        record_result("get_customer", {"status": "error", "customer_id": "C1", "message": "CRM timeout"}, s)
        assert s.verified_customer_id is None, "only status 'found' verifies; an error result that echoes an ID must not"
        record_result("get_customer", {"status": "found", "customer_id": "C1"}, s)
        assert s.verified_customer_id == "C1", "status 'found' sets verified_customer_id from the tool result"

        assert gate("lookup_order", {"order_id": "ORD-A"}, s) is None, "lookup_order is allowed once verified"
        assert gate("process_refund", {"order_id": "ORD-A", "amount": 10.0}, s), "rule 2: the order was not looked up yet"
        record_result("lookup_order", order_a, s)
        record_result("lookup_order", order_b, s)          # someone else's order ends up in state
        assert "ORD-A" in s.orders, "lookup_order results go into state.orders"
        assert gate("process_refund", {"order_id": "ORD-B", "amount": 10.0}, s), "rule 2: ORD-B belongs to another customer"

        assert gate("process_refund", {"order_id": "ORD-A", "amount": 0}, s), "rule 3: amount must be positive"
        assert gate("process_refund", {"order_id": "ORD-A", "amount": 950.0}, s), "rule 3: more than the order total"
        assert s.threshold_denials == 0, "rule 3 blocks must not count as threshold denials (check rule 3 before rule 4)"

        assert gate("process_refund", {"order_id": "ORD-A", "amount": 500.0}, s) is None, \
            "exactly $500 is within the limit (use >, not >=)"
        record_result("process_refund", {"status": "refunded", "order_id": "ORD-A", "amount": 500.0}, s)
        assert abs(s.refunded.get("ORD-A", 0) - 500.0) < 1e-9, "record refunded amounts per order"
        assert gate("process_refund", {"order_id": "ORD-A", "amount": 450.0}, s), \
            "rule 3: $500 already refunded + $450 exceeds the $900 total"

        s2 = CaseState(verified_customer_id="C1", orders={"ORD-A": order_a})
        msg = gate("process_refund", {"order_id": "ORD-A", "amount": 600.0}, s2)
        assert isinstance(msg, str) and msg.startswith("BLOCKED"), "rule 4: $600 is above the limit"
        assert "escalate_to_human" in msg and "threshold_exceeded" in msg, \
            "rule 4: deny with direction: name escalate_to_human and threshold_exceeded in the message"
        assert s2.threshold_denials == 1, "rule 4 increments threshold_denials"

        s4 = CaseState(verified_customer_id="C1", orders={"ORD-A": order_a})
        assert gate("process_refund", {"order_id": "ORD-A", "amount": 450.0}, s4) is None, "a single $450 refund is allowed"
        record_result("process_refund", {"status": "refunded", "order_id": "ORD-A", "amount": 450.0}, s4)
        msg = gate("process_refund", {"order_id": "ORD-A", "amount": 450.0}, s4)
        assert isinstance(msg, str) and "threshold_exceeded" in msg and s4.threshold_denials == 1, \
            "rule 4: a second $450 makes $900 in refunds, above the $500 limit (check the running total, not one call)"

        ticket = close_case(s2)
        assert ticket == {"escalation_reason": "threshold_exceeded", "customer_id": "C1", "order_ids": ["ORD-A"]}, \
            f"close_case should fail closed with the case facts, got {ticket}"
        assert s2.escalated and close_case(s2) is None, "after auto-escalating, a second close_case is a no-op"

        s3 = CaseState(verified_customer_id="C1", orders={"ORD-A": order_a}, threshold_denials=1)
        record_result("escalate_to_human", {"status": "queued"}, s3)
        assert s3.escalated and close_case(s3) is None, "no auto-escalation when the model already escalated"
        assert close_case(CaseState()) is None, "nothing blocked -> nothing to escalate"
    _run_check(3, checks)


check_ex3()

### Exercise 4 — The handoff as a strict tool, plus a completeness validator

Two parts. `HandoffSummary` and `CaseState` come from the setup cell.

**(a)** `build_escalate_tool(model_cls)` returns the tool definition: `name` `"escalate_to_human"`, `strict: True`, `input_schema` = `model_cls.model_json_schema()`, and a `description` of at least 25 words that tells the model the human will **not** see the conversation (use the word "conversation") and must get a complete summary.

**(b)** `handoff_problems(h, state)` returns a list of problem **codes** (any order, no duplicates):

| code | when |
|---|---|
| `identity_mismatch` | `h.identity_verified` disagrees with `state.verified_customer_id is not None` |
| `wrong_customer_id` | `h.customer_id` is set and differs from `state.verified_customer_id` |
| `no_actions` | the reason is not `customer_requested_human` and `h.actions_taken` is empty |
| `missing_open_question` | the reason is `policy_gap` and `h.open_questions` is empty |
| `missing_amount` | the reason is `threshold_exceeded` and `h.amount_at_stake_usd` is `None` |
| `empty_recommendation` | `h.recommended_action` is empty or only whitespace |

[Theory: 9.6](README.md#96-structured-handoff-protocols)

In [ ]:
def build_escalate_tool(model_cls) -> dict:
    # TODO: name, description (>= 25 words, mentions the conversation), strict, input_schema
    raise NotImplementedError


def handoff_problems(h: HandoffSummary, state: CaseState) -> list[str]:
    # TODO: return the problem codes from the table
    raise NotImplementedError

In [ ]:
def check_ex4():
    def checks():
        tool = build_escalate_tool(HandoffSummary)
        assert isinstance(tool, dict), "build_escalate_tool must return a dict"
        assert tool.get("name") == "escalate_to_human", "name must be escalate_to_human"
        assert tool.get("strict") is True, "set strict: True so the handoff shape is guaranteed"
        schema = tool.get("input_schema", {})
        assert schema.get("additionalProperties") is False and "escalation_reason" in schema.get("properties", {}), \
            "input_schema must be HandoffSummary.model_json_schema()"
        assert set(schema.get("required", [])) == set(schema["properties"]), "every property must be required (nullable, not optional)"
        desc = tool.get("description", "")
        assert len(desc.split()) >= 25, f"description has {len(desc.split())} words; write at least 25"
        assert "conversation" in desc.lower(), "tell the model the human will not see the conversation"

        base = {"escalation_reason": "policy_gap", "reason_detail": "Competitor price match requested.",
                "customer_id": "C1", "identity_verified": True, "order_ids": ["ORD-1"],
                "issue_summary": "Wants $54 back to match a competitor.", "root_cause": None,
                "facts_verified": ["ORD-1 paid $179.99"], "customer_claims": ["Competitor lists $125.99"],
                "actions_taken": [{"step": "lookup_order(ORD-1)", "outcome": "Delivered, $179.99"}],
                "offers_made": ["Return for refund: declined"], "amount_at_stake_usd": 54.0,
                "recommended_action": "Decide on a one-off goodwill adjustment.", "recommendation_rationale": "No policy basis.",
                "urgency": "normal", "open_questions": ["Price-match as an exception?"]}

        def problems(state=None, **overrides):
            h = HandoffSummary.model_validate({**base, **overrides})
            out = handoff_problems(h, state or CaseState(verified_customer_id="C1"))
            assert isinstance(out, list), "handoff_problems must return a list"
            return set(out)

        assert problems() == set(), f"a complete, consistent handoff should have no problems, got {problems()}"
        assert problems(state=CaseState()) == {"identity_mismatch", "wrong_customer_id"}, \
            f"unverified state vs a handoff claiming C1 verified: expected identity_mismatch + wrong_customer_id, got {problems(state=CaseState())}"
        assert problems(customer_id="C9") == {"wrong_customer_id"}, f"C9 is not the verified customer, got {problems(customer_id='C9')}"
        assert problems(actions_taken=[]) == {"no_actions"}, "a policy gap with nothing tried needs no_actions"
        human = problems(escalation_reason="customer_requested_human", actions_taken=[], open_questions=[], amount_at_stake_usd=None)
        assert human == set(), f"an immediate escalation on request may have no actions, open questions or amount; got {human}"
        assert problems(open_questions=[]) == {"missing_open_question"}, "a policy gap needs an open question"
        assert problems(escalation_reason="threshold_exceeded", amount_at_stake_usd=None, open_questions=[]) == {"missing_amount"}, \
            "a threshold escalation needs the amount at stake (and doesn't need an open question)"
        assert problems(recommended_action="   ") == {"empty_recommendation"}, "whitespace is an empty recommendation"
    _run_check(4, checks)


check_ex4()

### Exercise 5 — Per-segment auto-accept thresholds from labeled data

Implement `segment_thresholds(records, target=0.98, min_n=100)`. Each record is `{"doc_type": str, "field": str, "confidence": float, "correct": bool}` from a labeled validation set. Return a dict keyed by `(doc_type, field)` for **every** segment present in `records`, whose value is that segment's threshold:

- Try candidate thresholds `t` in **ascending** order of the segment's distinct confidence values.
- The auto-accepted set for `t` is every record in the segment with `confidence >= t`.
- If that set has fewer than `min_n` records, stop: the segment's value is `None` (keep it on human review).
- Otherwise, if its accuracy is `>= target`, the segment's value is `t`.
- If no `t` qualifies, the value is `None`.

Calibrate each segment on its own records only: one global threshold is the anti-pattern this exercise is about.

[Theory: 9.7](README.md#97-confidence-calibration)

In [ ]:
def segment_thresholds(records: list[dict], target: float = 0.98, min_n: int = 100) -> dict:
    # TODO: group by (doc_type, field), then pick the lowest threshold that meets target with >= min_n accepted
    raise NotImplementedError

In [ ]:
def check_ex5():
    def make(doc_type, field_name, conf, n, n_correct):
        return [{"doc_type": doc_type, "field": field_name, "confidence": conf, "correct": i < n_correct} for i in range(n)]

    def checks():
        records = (
            make("invoice", "total", 0.99, 100, 100) + make("invoice", "total", 0.80, 50, 40) + make("invoice", "total", 0.60, 50, 20)
            + make("receipt", "total", 0.99, 80, 80) + make("receipt", "total", 0.70, 100, 60)
            + make("invoice", "vendor", 0.85, 300, 297) + make("invoice", "vendor", 0.50, 20, 0)
            + make("contract", "total", 0.95, 100, 100) + make("contract", "total", 0.50, 100, 50)
        )
        snapshot = json.dumps(records)
        got = segment_thresholds(records, target=0.98, min_n=100)
        assert isinstance(got, dict), "return a dict keyed by (doc_type, field)"
        expected_keys = {("invoice", "total"), ("receipt", "total"), ("invoice", "vendor"), ("contract", "total")}
        assert set(got) == expected_keys, f"one entry per segment present; expected {sorted(expected_keys)}, got {sorted(got)}"
        assert got[("invoice", "total")] == 0.99, f"invoice/total: only the 0.99 band reaches 98%, got {got[('invoice', 'total')]}"
        assert got[("invoice", "vendor")] == 0.85, f"invoice/vendor: 0.85 gives 297/300 = 99%, got {got[('invoice', 'vendor')]}"
        assert got[("receipt", "total")] is None, \
            f"receipt/total: the 0.99 band is 100% accurate but has only 80 items (< min_n=100), so keep it on review; got {got[('receipt', 'total')]}"
        assert got[("contract", "total")] == 0.95, \
            f"contract/total: exactly min_n (100) accepted items is enough, got {got[('contract', 'total')]}"
        loose = segment_thresholds(records, target=0.80, min_n=100)
        assert loose[("invoice", "total")] == 0.60 and loose[("receipt", "total")] is None, \
            f"with target 0.80: invoice/total -> 0.60 (160/200 = 80%), receipt/total -> None (140/180 = 78%, then too few); got {loose}"
        assert json.dumps(records) == snapshot, "do not mutate the records"
    _run_check(5, checks)


check_ex5()

### Exercise 6 — Segment accuracy, stratified audit allocation and the rule of three

Four small functions.

1. `segment_accuracy(records)` → `{doc_type: {"n": int, "accuracy": float}}` over records shaped like Exercise 5's.
2. `segments_to_automate(records, target=0.97, min_n=200)` → a **sorted** list of the document types whose accuracy is `>= target` **and** whose `n >= min_n`. Use per-segment numbers, never the aggregate.
3. `allocate_audit(strata_sizes, rate, default_rate=0.02, min_per_stratum=30)` → `{stratum: k}` where `k = min(size, max(min_per_stratum, round(size * rate.get(stratum, default_rate))))`. Small strata get at least `min_per_stratum` items, but never more than they have.
4. `max_error_if_zero_found(n)` → the rule-of-three 95% upper bound, `3 / n`.

[Theory: 9.8](README.md#98-segment-accuracy-and-stratified-sampling)

In [ ]:
def segment_accuracy(records: list[dict]) -> dict:
    # TODO
    raise NotImplementedError


def segments_to_automate(records: list[dict], target: float = 0.97, min_n: int = 200) -> list[str]:
    # TODO
    raise NotImplementedError


def allocate_audit(strata_sizes: dict, rate: dict, default_rate: float = 0.02, min_per_stratum: int = 30) -> dict:
    # TODO
    raise NotImplementedError


def max_error_if_zero_found(n: int) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6():
    def make(doc_type, n, n_correct):
        return [{"doc_type": doc_type, "field": "total", "confidence": 0.9, "correct": i < n_correct} for i in range(n)]

    def checks():
        records = make("typed_invoice", 9000, 8910) + make("contract", 500, 490) + make("handwritten_receipt", 500, 300) + make("fax", 100, 100)
        acc = segment_accuracy(records)
        assert isinstance(acc, dict) and set(acc) == {"typed_invoice", "contract", "handwritten_receipt", "fax"}, \
            "one entry per doc_type"
        assert acc["handwritten_receipt"]["n"] == 500 and abs(acc["handwritten_receipt"]["accuracy"] - 0.60) < 1e-9, \
            f"handwritten_receipt should be n=500, accuracy 0.60; got {acc['handwritten_receipt']}"
        assert abs(acc["typed_invoice"]["accuracy"] - 0.99) < 1e-9, "typed_invoice accuracy should be 0.99"

        auto = segments_to_automate(records, target=0.97, min_n=200)
        assert auto == ["contract", "typed_invoice"], \
            (f"expected ['contract', 'typed_invoice'] (receipts fail the target; fax is 100% but only 100 samples), got {auto}. "
             "The aggregate is 97% too, which is exactly why you must not use it.")

        sizes = {"typed_invoice": 9000, "contract": 500, "handwritten_receipt": 500, "fax": 12}
        alloc = allocate_audit(sizes, rate={"handwritten_receipt": 0.10})
        assert alloc == {"typed_invoice": 180, "contract": 30, "handwritten_receipt": 50, "fax": 12}, \
            (f"expected typed 180 (2%), contract 30 (minimum), receipts 50 (10%), fax 12 (all it has), got {alloc}")

        assert abs(max_error_if_zero_found(300) - 0.01) < 1e-12 and abs(max_error_if_zero_found(50) - 0.06) < 1e-12, \
            "rule of three: 3 / n"
    _run_check(6, checks)


check_ex6()

### Optional live exercise (live — calls the API, ~$0.02, not scored)

Write your own escalation prompt in `MY_ESCALATION_PROMPT` (criteria plus a few boundary examples, as in [README 9.2](README.md#92-escalation-patterns)) and see whether Claude makes the same decisions as your `route_turn` would on six new turns. If a turn misses, add an example for that boundary and rerun.

In [ ]:
class Decision(BaseModel):
    decision: Literal["resolve", "ask", "escalate"]
    reason: Literal["customer_requested_human", "policy_gap", "cannot_make_progress", "threshold_exceeded", "none"]


MY_ESCALATION_PROMPT = """TODO: write your escalation criteria and 3-4 boundary examples here (see README 9.2).
Then add: classify the latest customer turn as resolve, ask or escalate (with a reason; "none" otherwise)."""

TURNS = [  # (context, message, the decision your route_turn would make)
    ("Verified; order inside the replacement window.", "This kettle is junk!! Third one this year!!", ("resolve", None)),
    ("Verified. Policy: refunds go to the original card; it says nothing about gift cards.", "Refund it as a gift card for my niece instead.", ("escalate", "policy_gap")),
    ("get_customer returned multiple_matches (2). No identifier requested yet.", "It's John Smith, cancel my order.", ("ask", None)),
    ("The agent offered a replacement.", "No. Put me through to a person.", ("escalate", "customer_requested_human")),
    ("Verified. process_refund returned BLOCKED: refunds over $500 need human approval.", "Just refund the $820, please.", ("escalate", "threshold_exceeded")),
    ("Verified. Damage with a photo is covered.", "Here's the photo of the cracked screen. New one please.", ("resolve", None)),
]

if not HAS_KEY:
    print("skipped (live): no ANTHROPIC_API_KEY")
elif MY_ESCALATION_PROMPT.startswith("TODO"):
    print("Write MY_ESCALATION_PROMPT first.")
else:
    hits = 0
    for context, message, expected in TURNS:
        r = client.messages.parse(model=MODEL, max_tokens=2048, system=MY_ESCALATION_PROMPT, output_format=Decision,
                                  messages=[{"role": "user", "content": f"<context>{context}</context>\n<customer>{message}</customer>"}])
        d = r.parsed_output
        got = ("error", r.stop_reason) if d is None else (d.decision, None if d.reason == "none" else d.reason)
        hits += got == expected
        print(f"{'✓' if got == expected else '✗'} expected {expected}, got {got}: {message}")
    print(f"{hits}/{len(TURNS)} correct")

## Part C — Architecture scenario

**Scenario.** An insurer is automating intake for home-insurance claims. About 3,000 claims a day arrive as structured web forms (70%), scanned paper forms (20%) and handwritten letters with photos (10%). Claude extracts: policy number, claimant name, incident date, damage category, a list of claimed items with amounts, the stated total, and a short description. Claims **under $2,000** are paid automatically when every check passes; everything else goes to an adjuster. The operations team has **three reviewers**, each able to check about 150 flagged fields a day. Leadership quotes a pilot result: "96% field accuracy, so we can cut review to almost nothing."

Design the human-review side of this system. Cover:

1. How you decide which segments and fields can be auto-accepted (data, method, thresholds).
2. Routing rules: what goes to a reviewer, and what the reviewer sees.
3. How the 450 daily review slots are prioritized when more is flagged than they can handle.
4. How you keep measuring the error rate of auto-accepted output after launch, and catch new error patterns.
5. Where the $2,000 auto-pay rule and other hard limits live, and what happens when a claim is blocked.
6. The handoff from the pipeline to an adjuster, and the metrics you report to leadership.

Write your answer in the cell below (about 250–350 words).

*Your answer:*

<details><summary>Rubric: what a strong answer contains (open after you have written yours)</summary>

- **Challenge the 96%.** Aggregate accuracy can hide failing segments. Measure accuracy per **channel/document type × field** on a labeled validation set that matches production (web forms, scans, handwritten letters; every field), with enough samples per segment.
- **Calibrated, per-segment thresholds.** Ask for field-level confidence (and a source quote), build reliability tables per field and segment, pick each auto-accept threshold for a target accuracy with a minimum sample size, verify on a held-out split. Handwritten letters likely stay on full review at first. Recalibrate on any model, prompt, schema or document-mix change.
- **Routing:** validation failures after a retry; contradictions recomputed in code (stated total vs the sum of items; incident date after the claim date); critical fields below threshold, `"unclear"` or `null` (for example a missing policy number, looked up in the policy system in code, not guessed). Reviewers see **only the flagged fields** next to their source snippets, and both values for conflicts.
- **Capacity:** reserve slots for the audit sample first, then sort by P(error) (from calibration) × cost of error (claim amount, auto-pay exposure). Long tails wait; no first-in-first-out; never drop the audit to save time.
- **Ongoing measurement:** a **stratified random sample of auto-accepted** items by channel × confidence band (oversample small or risky strata, weight back by volume when reporting), sized with the rule of three (about 300 clean items per stratum to claim < 1%). Reviewer findings feed back into calibration; a stratum above target goes back on full review.
- **Hard limits in code:** the < $2,000 auto-pay rule, "all checks passed" and duplicate-claim checks are deterministic gates before payment, not prompt instructions. A blocked claim fails closed to an adjuster queue.
- **Adjuster handoff:** a structured, self-contained summary (claim and policy IDs, verified facts vs claimant statements, amounts, what was checked, why it was routed, recommended action, open questions), with hard facts filled by code.
- **Metrics:** per-segment accuracy and audited error rate, auto-accept share, review backlog and age, reviewer agreement (labels quality), routing precision/recall, false auto-pay rate. Bonus: least privilege (the extraction step cannot trigger payment itself).

</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_RESULT if QUIZ_RESULT else (0, 10)
ex_total = 6
ex_passed = sum(RESULTS.get(n, False) for n in range(1, ex_total + 1))
quiz_pct, ex_pct = quiz_correct / quiz_total, ex_passed / ex_total
overall = (quiz_pct + ex_pct) / 2

print(f"Quiz       : {quiz_correct}/{quiz_total} ({quiz_pct:.0%})" + ("" if QUIZ_RESULT else "  <- run the quiz grader"))
print(f"Exercises  : {ex_passed}/{ex_total} passed ({ex_pct:.0%})   " +
      " ".join(f"ex{n}:{'✓' if RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall    : {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers and explanations</summary>

**Q1: C.** An explicit request for a human is honored immediately. What the agent already knows (the order number) goes into the handoff, but it doesn't start new work to fill it in. A and B investigate first and ignore the request. D applies the acknowledge → offer → reiterate pattern, which is for frustration *without* a request for a person.

**Q2: A.** The policy is silent on this specific request, so it is the business's decision. B stretches the policy ("close enough"), C refuses on a technicality, and D routes on self-rated confidence, which is not a valid trigger and decides something the agent has no authority to decide.

**Q3: D.** A gate is only deterministic if its state comes from the source of truth: tool results. A and C are still probabilistic, and B still trusts model text, so the same attack works with the new sentence.

**Q4: B.** Fail closed: the harness knows a block happened and knows whether an escalation followed, so it can guarantee the human gets the case. A and C lower the miss rate but cannot make it zero. D changes the business rule to hide the symptom.

**Q5: C.** The docs say it directly: custom tools are executed by your application and are not governed by permission policies. Your code receives `agent.custom_tool_use` and decides. A is wrong twice: `always_ask` has no interactive-only restriction, and `auto` is not a human checkpoint. B's `configs` entries name tools of the toolset, not your custom tools. D removes structure and makes control worse.

**Q6: B.** Confidence is a ranking until it is calibrated, and calibration differs by field and segment. A treats the score as a probability. C is still one uncalibrated global threshold. D changes the model's numbers without measuring anything.

**Q7: D.** The rule of three: with zero observed errors in n items, the 95% upper bound on the true rate is about 3/n. A, B and C all overstate what 50 clean samples prove.

**Q8: A.** Scarce attention goes where errors cost most, and the audit sample is never dropped because it is the only unbiased measure of the automated path. B removes that measurement and ignores cost. C auto-accepts fields that failed calibration. D wastes slots on low-risk items (randomness is for the audit, not the work queue).

**Q9: B.** A contradiction in the source is a routing signal regardless of confidence: only a person (or the business) can decide which value is right. A trusts a confident value the document itself contradicts. C lets the model resolve it. D can never succeed for a document that contradicts itself and pressures the model to change a stated value.

**Q10: D.** Resolve the in-scope part, escalate only the part that needs a human, and tell the human what was already done so they don't redo or contradict it. A escalates a case the agent can resolve. B refuses something a human can do. C pushes the work back onto the customer.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def route_turn(turn: dict) -> tuple[str, str | None]:
    if turn.get("asks_human", False):
        return ("escalate", "customer_requested_human")
    if turn.get("matches", 1) != 1:
        if turn.get("id_requests", 0) >= 2:
            return ("escalate", "cannot_make_progress")
        return ("ask", None)
    if turn.get("policy_covers", True) is not True:
        return ("escalate", "policy_gap")
    if turn.get("gate_blocked", False):
        return ("escalate", "threshold_exceeded")
    if turn.get("tool_failures", 0) >= 3 or turn.get("rejected_offers", 0) >= 2:
        return ("escalate", "cannot_make_progress")
    return ("resolve", None)
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def customer_lookup_result(matches: list[dict]) -> dict:
    if len(matches) == 1:
        c = matches[0]
        return {"status": "found", "customer_id": c["customer_id"], "name": c["name"], "order_ids": list(c["order_ids"])}
    if len(matches) > 1:
        return {"status": "multiple_matches", "count": len(matches), "ask_for_one_of": ["email", "phone", "order_number"]}
    return {"status": "not_found", "hint": "Ask the customer to check the spelling or give their email."}
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def gate(tool: str, args: dict, state: CaseState) -> str | None:
    if tool in ("lookup_order", "process_refund") and state.verified_customer_id is None:
        return "BLOCKED: verify the customer with get_customer first (exactly one match)."
    if tool == "process_refund":
        order = state.orders.get(args.get("order_id"))
        if order is None or order["customer_id"] != state.verified_customer_id:
            return "BLOCKED: refund only an order that lookup_order returned for the verified customer."
        amount = args["amount"]
        already = state.refunded.get(order["order_id"], 0.0)
        if amount <= 0 or amount + already > order["total"]:
            return f"BLOCKED: the amount must be positive and at most {order['total'] - already:.2f}."
        if amount + already > REFUND_LIMIT_USD:                 # running total, not one call
            state.threshold_denials += 1
            return (f"BLOCKED by policy: refunds over ${REFUND_LIMIT_USD:.0f} need human approval. "
                    "Call escalate_to_human with escalation_reason='threshold_exceeded' and a complete handoff.")
    return None


def record_result(tool: str, output: dict, state: CaseState) -> None:
    if tool == "get_customer" and output.get("status") == "found":
        state.verified_customer_id = output["customer_id"]
    elif tool == "lookup_order" and "order_id" in output:
        state.orders[output["order_id"]] = output
    elif tool == "process_refund" and output.get("status") == "refunded":
        state.refunded[output["order_id"]] = state.refunded.get(output["order_id"], 0.0) + output["amount"]
    elif tool == "escalate_to_human":
        state.escalated = True


def close_case(state: CaseState) -> dict | None:
    if state.threshold_denials > 0 and not state.escalated:
        state.escalated = True
        return {"escalation_reason": "threshold_exceeded", "customer_id": state.verified_customer_id,
                "order_ids": sorted(state.orders)}
    return None
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def build_escalate_tool(model_cls) -> dict:
    return {
        "name": "escalate_to_human",
        "description": (
            "Hand the case to a human support specialist. The specialist will NOT see this conversation, "
            "only this summary, so make it complete and self-contained: who the customer is, what they want, "
            "what you verified and tried, the amount at stake, your recommendation and the open questions. "
            "After calling it, tell the customer what happens next."
        ),
        "strict": True,
        "input_schema": model_cls.model_json_schema(),
    }


def handoff_problems(h: HandoffSummary, state: CaseState) -> list[str]:
    problems = []
    if h.identity_verified != (state.verified_customer_id is not None):
        problems.append("identity_mismatch")
    if h.customer_id and h.customer_id != state.verified_customer_id:
        problems.append("wrong_customer_id")
    if h.escalation_reason != "customer_requested_human" and not h.actions_taken:
        problems.append("no_actions")
    if h.escalation_reason == "policy_gap" and not h.open_questions:
        problems.append("missing_open_question")
    if h.escalation_reason == "threshold_exceeded" and h.amount_at_stake_usd is None:
        problems.append("missing_amount")
    if not h.recommended_action.strip():
        problems.append("empty_recommendation")
    return problems
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def segment_thresholds(records: list[dict], target: float = 0.98, min_n: int = 100) -> dict:
    segments = defaultdict(list)
    for r in records:
        segments[(r["doc_type"], r["field"])].append(r)
    out = {}
    for key, seg in segments.items():
        out[key] = None
        for t in sorted({r["confidence"] for r in seg}):
            accepted = [r["correct"] for r in seg if r["confidence"] >= t]
            if len(accepted) < min_n:
                break
            if sum(accepted) / len(accepted) >= target:
                out[key] = t
                break
    return out
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def segment_accuracy(records: list[dict]) -> dict:
    groups = defaultdict(list)
    for r in records:
        groups[r["doc_type"]].append(r["correct"])
    return {d: {"n": len(v), "accuracy": sum(v) / len(v)} for d, v in groups.items()}


def segments_to_automate(records: list[dict], target: float = 0.97, min_n: int = 200) -> list[str]:
    return sorted(d for d, s in segment_accuracy(records).items() if s["accuracy"] >= target and s["n"] >= min_n)


def allocate_audit(strata_sizes: dict, rate: dict, default_rate: float = 0.02, min_per_stratum: int = 30) -> dict:
    return {s: min(size, max(min_per_stratum, round(size * rate.get(s, default_rate)))) for s, size in strata_sizes.items()}


def max_error_if_zero_found(n: int) -> float:
    return 3 / n
```

</details>

<details><summary>Optional live exercise — a prompt that works</summary>

```text
You classify the latest customer turn for Acme Home support. Decide resolve, ask or escalate.
Escalate when, and only when: (1) the customer explicitly asks for a person (reason customer_requested_human);
(2) policy does not cover the specific request or is ambiguous about it (policy_gap); (3) identity is still
unverified after two requests, a tool keeps failing, or two valid offers were rejected (cannot_make_progress);
(4) a tool result says the action needs human approval (threshold_exceeded).
Ask when several accounts match and no identifier has been requested twice yet.
Otherwise resolve. Frustration, a long story or your own uncertainty are not reasons to escalate.
Use reason "none" unless you escalate.

<examples>
<example><customer>My blender broke after 3 days, ridiculous!</customer><decision>resolve</decision></example>
<example><customer>Can you match a competitor's price?</customer><decision>escalate: policy_gap</decision></example>
<example><customer>(3 accounts match) It's Ana, refund me.</customer><decision>ask</decision></example>
<example><customer>No replacement. Get me a manager.</customer><decision>escalate: customer_requested_human</decision></example>
</examples>
```

</details>

<details><summary>Part C — sample answer outline</summary>

Label ~300+ claims per channel × field and build reliability tables. Web forms: auto-accept fields whose calibrated threshold meets 98% with enough samples; scans: selected fields only; handwritten letters: full review at launch. "96%" is an aggregate and is rejected as a basis. Routing: validation failures, total vs item-sum conflicts (recomputed in code), date contradictions, `null`/`"unclear"` critical fields, and fields below threshold go to review, field-level, with source snippets and both values. Daily capacity (450): reserve ~60 audit slots for a stratified random sample of auto-accepted claims (channel × confidence band, oversampling scans and letters, weighted back by volume, ~300 clean items per stratum before claiming < 1%), then sort the rest by P(error) × claim amount. The < $2,000 auto-pay rule and duplicate checks are code gates before payment; blocked claims fail closed to adjusters with a structured handoff (IDs, verified facts vs claimant statements, amounts, checks done, reason, recommendation, open questions). Report per-segment accuracy, audited error rates, auto-accept share, backlog age, reviewer agreement and false auto-pay rate; recalibrate on every model or prompt change.

</details>

## Done

- Below 72%? Reread the README sections behind your ✗ marks, rerun the matching practice cells, and retake.
- Passed? Mark the chapter **Status** as done in the [README](README.md).

Next chapter: [Chapter 10 — Error Handling in Multi-agent Systems](../10-multi-agent-error-handling/README.md)